# Обучение LoRA-адаптеров для квиз-пайплайна

Двухстадийный пайплайн:
```
TEXT → [Fact Extraction LoRA] → FACTS (JSON) → [Quiz Generation LoRA] → QUIZ
```

- **Base модель:** Mistral 7B Instruct v0.3
- **Метод:** QLoRA (4-bit NF4)
- **Обучение:** RTX 3080 Ti (12GB VRAM)
- **Инференс:** Tesla T4 (16GB VRAM)

## 1. Setup

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 trl>=0.8.0 datasets accelerate scipy

In [ ]:
import gc
import json
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Пути
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data" / "datasets"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# Гиперпараметры
MAX_SEQ_LENGTH = 4096
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8  # effective batch = 16
WARMUP_RATIO = 0.05

## 2. Загрузка данных

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    data = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


# Загрузка датасетов
ext_train = load_jsonl(DATA_DIR / "extraction_train.jsonl")
ext_val = load_jsonl(DATA_DIR / "extraction_val.jsonl")
gen_train = load_jsonl(DATA_DIR / "generation_train.jsonl")
gen_val = load_jsonl(DATA_DIR / "generation_val.jsonl")

print(f"Extraction — train: {len(ext_train)}, val: {len(ext_val)}")
print(f"Generation — train: {len(gen_train)}, val: {len(gen_val)}")

In [ ]:
# Статистика длин (в символах) для оценки max_seq_length
import numpy as np


def total_chars(sample):
    return sum(len(m["content"]) for m in sample["messages"])


ext_lengths = [total_chars(s) for s in ext_train]
gen_lengths = [total_chars(s) for s in gen_train]

print("Extraction (символы):")
print(f"  median={np.median(ext_lengths):.0f}, p95={np.percentile(ext_lengths, 95):.0f}, max={max(ext_lengths)}")
print("Generation (символы):")
print(f"  median={np.median(gen_lengths):.0f}, p95={np.percentile(gen_lengths, 95):.0f}, max={max(gen_lengths)}")

## 3. Модель и токенизатор

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


def load_base_model():
    """Загрузка base модели с 4-bit квантизацией."""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
    model.config.use_cache = False
    return model


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Chat template: {'yes' if tokenizer.chat_template else 'no'}")

In [ ]:
# Проверка длин в токенах
def count_tokens(sample):
    text = tokenizer.apply_chat_template(sample["messages"], tokenize=False)
    return len(tokenizer.encode(text))


ext_tok_lengths = [count_tokens(s) for s in ext_train[:100]]
gen_tok_lengths = [count_tokens(s) for s in gen_train[:100]]

print("Extraction (токены, первые 100):")
print(
    f"  median={np.median(ext_tok_lengths):.0f}, p95={np.percentile(ext_tok_lengths, 95):.0f}, max={max(ext_tok_lengths)}"
)
print("Generation (токены, первые 100):")
print(
    f"  median={np.median(gen_tok_lengths):.0f}, p95={np.percentile(gen_tok_lengths, 95):.0f}, max={max(gen_tok_lengths)}"
)

# Если p95 сильно меньше MAX_SEQ_LENGTH — можно уменьшить для экономии VRAM
suggested = max(np.percentile(ext_tok_lengths, 95), np.percentile(gen_tok_lengths, 95))
suggested = int(np.ceil(suggested / 256) * 256)  # округление до 256
print(f"\nРекомендуемый max_seq_length: {suggested} (текущий: {MAX_SEQ_LENGTH})")

## 4. LoRA конфигурация

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Target modules: {lora_config.target_modules}")

In [ ]:
def make_training_args(output_dir: str, num_train_samples: int) -> TrainingArguments:
    steps_per_epoch = num_train_samples // (BATCH_SIZE * GRAD_ACCUM)
    eval_steps = max(1, steps_per_epoch // 4)

    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="steps",
        save_steps=eval_steps,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        optim="paged_adamw_8bit",
        max_grad_norm=0.3,
        seed=42,
    )

In [ ]:
def formatting_func(example):
    """Форматирование сэмпла через chat template."""
    return tokenizer.apply_chat_template(example["messages"], tokenize=False)


def train_lora(train_data, val_data, output_dir, task_name):
    """Обучение одного LoRA-адаптера."""
    print(f"\n{'=' * 60}")
    print(f"Обучение: {task_name}")
    print(f"Train: {len(train_data)}, Val: {len(val_data)}")
    print(f"{'=' * 60}\n")

    # Загрузка модели
    model = load_base_model()
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # HF Datasets
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)

    # Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=make_training_args(output_dir + "_checkpoints", len(train_data)),
        formatting_func=formatting_func,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        tokenizer=tokenizer,
    )

    # Обучение
    trainer.train()

    # Сохранение лучшего адаптера
    model.save_pretrained(output_dir)
    print(f"\nАдаптер сохранён: {output_dir}")

    # Очистка VRAM
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return output_dir

## 5. Обучение — Fact Extraction LoRA

In [ ]:
extraction_dir = str(MODELS_DIR / "extraction_lora")
train_lora(ext_train, ext_val, extraction_dir, "Fact Extraction")

## 6. Обучение — Quiz Generation LoRA

In [ ]:
generation_dir = str(MODELS_DIR / "generation_lora")
train_lora(gen_train, gen_val, generation_dir, "Quiz Generation")

## 7. Оценка

Загружаем base + адаптер и проверяем качество на val-сете.

In [ ]:
def load_model_with_adapter(adapter_path: str):
    """Загрузка base модели с LoRA-адаптером для инференса."""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return model


def generate_response(model, messages: list[dict], max_new_tokens: int = 2048) -> str:
    """Генерация ответа модели."""
    # Берём только system + user для промпта
    prompt_messages = [m for m in messages if m["role"] != "assistant"]
    prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    return response.strip()

In [ ]:
def evaluate_extraction(model, val_data, n_samples=50):
    """Оценка качества fact extraction."""
    import random

    samples = random.sample(val_data, min(n_samples, len(val_data)))

    valid_json = 0
    total_facts = []
    type_coverage = set()

    for sample in samples:
        response = generate_response(model, sample["messages"])
        try:
            facts = json.loads(response)
            if isinstance(facts, list) and len(facts) > 0:
                valid_json += 1
                total_facts.append(len(facts))
                for f in facts:
                    if "type" in f:
                        type_coverage.add(f["type"])
        except (json.JSONDecodeError, TypeError):
            pass

    n = len(samples)
    print(f"Extraction evaluation ({n} samples):")
    print(f"  Valid JSON: {valid_json}/{n} ({100 * valid_json / n:.1f}%)")
    if total_facts:
        print(f"  Avg facts per sample: {np.mean(total_facts):.1f}")
    print(f"  Fact types seen: {sorted(type_coverage)}")


def evaluate_generation(model, val_data, n_samples=50):
    """Оценка качества quiz generation."""
    import random

    samples = random.sample(val_data, min(n_samples, len(val_data)))

    valid_json = 0
    total_questions = []
    question_types = {}
    has_options = 0

    for sample in samples:
        response = generate_response(model, sample["messages"])
        try:
            quiz = json.loads(response)
            if isinstance(quiz, dict) and "questions" in quiz:
                questions = quiz["questions"]
                if len(questions) > 0:
                    valid_json += 1
                    total_questions.append(len(questions))
                    for q in questions:
                        t = q.get("type", "unknown")
                        question_types[t] = question_types.get(t, 0) + 1
                        if "options" in q:
                            has_options += 1
        except (json.JSONDecodeError, TypeError):
            pass

    n = len(samples)
    print(f"Generation evaluation ({n} samples):")
    print(f"  Valid JSON: {valid_json}/{n} ({100 * valid_json / n:.1f}%)")
    if total_questions:
        print(f"  Avg questions per quiz: {np.mean(total_questions):.1f}")
    print(f"  Question types: {question_types}")
    print(f"  Questions with options: {has_options}")

In [ ]:
# Оценка Extraction
print("Загрузка extraction модели...")
ext_model = load_model_with_adapter(extraction_dir)
evaluate_extraction(ext_model, ext_val)

del ext_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Оценка Generation
print("Загрузка generation модели...")
gen_model = load_model_with_adapter(generation_dir)
evaluate_generation(gen_model, gen_val)

del gen_model
gc.collect()
torch.cuda.empty_cache()

## 8. Full Pipeline Inference

Полный пайплайн: text → facts → quiz. Один base, два адаптера.

In [ ]:
# Системные промпты (совпадают с prepare_ft_datasets.py)
EXTRACTION_SYSTEM = (
    "Ты — AI-ассистент для извлечения фактов из учебных текстов. "
    "Извлеки ключевые факты в формате JSON-массива. Каждый факт содержит поля: "
    "question (вопрос по факту), answer (краткий ответ), type (один из: "
    "person, date, term, number, location, event, definition, process)."
)

GENERATION_SYSTEM = (
    "Ты — AI-ассистент для создания образовательных квизов. "
    "На основе списка фактов создай квиз в формате JSON с полем questions. "
    "Типы вопросов: single_choice (с options), multiple_choice (answer — массив, с options), "
    "short_answer (без options, краткий ответ)."
)


class QuizPipeline:
    def __init__(self, extraction_adapter: str, generation_adapter: str):
        self.tokenizer = tokenizer
        self.base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        self.extraction_adapter = extraction_adapter
        self.generation_adapter = generation_adapter

    def _generate(self, model, messages, max_new_tokens=2048):
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.3,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        return self.tokenizer.decode(outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

    def extract_facts(self, text: str) -> list[dict]:
        model = PeftModel.from_pretrained(self.base_model, self.extraction_adapter)
        model.eval()
        messages = [
            {"role": "system", "content": EXTRACTION_SYSTEM},
            {"role": "user", "content": text},
        ]
        response = self._generate(model, messages)
        del model
        self.base_model = self.base_model.base_model  # unwrap PEFT
        return json.loads(response)

    def generate_quiz(self, facts: list[dict]) -> dict:
        model = PeftModel.from_pretrained(self.base_model, self.generation_adapter)
        model.eval()
        facts_str = json.dumps(facts, ensure_ascii=False)
        messages = [
            {"role": "system", "content": GENERATION_SYSTEM},
            {"role": "user", "content": f"Факты:\n{facts_str}"},
        ]
        response = self._generate(model, messages)
        del model
        self.base_model = self.base_model.base_model  # unwrap PEFT
        return json.loads(response)

    def run(self, text: str) -> dict:
        print("Extracting facts...")
        facts = self.extract_facts(text)
        print(f"  Found {len(facts)} facts")

        print("Generating quiz...")
        quiz = self.generate_quiz(facts)
        print(f"  Generated {len(quiz.get('questions', []))} questions")

        return {"facts": facts, "quiz": quiz}

In [ ]:
# Демо на примерах из val-сета
pipeline = QuizPipeline(extraction_dir, generation_dir)

# Берём 3 текста из исходных данных
texts_path = BASE_DIR / "data" / "foxford_data" / "texts.jsonl"
demo_texts = []
with open(texts_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i in [0, 100, 500]:  # выбираем 3 текста из разных мест
            demo_texts.append(json.loads(line))

for idx, item in enumerate(demo_texts):
    print(f"\n{'=' * 60}")
    print(f"Демо {idx + 1}: {item['text'][:100]}...")
    print(f"{'=' * 60}")

    result = pipeline.run(item["text"])

    print(f"\nФакты ({len(result['facts'])}):")
    for f in result["facts"][:3]:
        print(f"  - [{f.get('type', '')}] {f.get('question', '')} → {f.get('answer', '')}")
    if len(result["facts"]) > 3:
        print(f"  ... и ещё {len(result['facts']) - 3}")

    print(f"\nКвиз ({len(result['quiz'].get('questions', []))} вопросов):")
    for q in result["quiz"].get("questions", [])[:3]:
        print(f"  - [{q.get('type', '')}] {q.get('question', '')}")
        if q.get("options"):
            print(f"    options: {q['options']}")
        print(f"    answer: {q.get('answer', '')}")

del pipeline
gc.collect()
torch.cuda.empty_cache()

## 9. Экспорт

Адаптеры сохранены в `models/extraction_lora/` и `models/generation_lora/`.

Для деплоя на Tesla T4:

In [ ]:
import os


def dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1024 / 1024


print("Размеры адаптеров:")
print(f"  extraction_lora: {dir_size_mb(extraction_dir):.1f} MB")
print(f"  generation_lora: {dir_size_mb(generation_dir):.1f} MB")

print("\nДля деплоя на Tesla T4 (16GB VRAM):")
print("  Base модель (4-bit): ~5 GB")
print("  Адаптеры: ~100 MB каждый")
print("  KV cache + overhead: ~3-5 GB")
print("  Итого: ~8-10 GB (влезает в T4 с запасом)")
print("\nФайлы для копирования на сервер:")
print(f"  {extraction_dir}/")
print(f"  {generation_dir}/")
print(f"  Base модель загружается из HuggingFace: {MODEL_ID}")

## 10. Push адаптеров на HuggingFace Hub

Для деплоя через Sphinx-сервис адаптеры загружаются в приватные репозитории HF.

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login

# Авторизация — запросит токен интерактивно
login()

api = HfApi()

# Названия репозиториев (замените "edium" на свой username/org)
EXTRACTION_REPO = "edium/sphinx-extraction-lora"
GENERATION_REPO = "edium/sphinx-generation-lora"

# Создание репозиториев (если не существуют)
api.create_repo(EXTRACTION_REPO, private=True, exist_ok=True)
api.create_repo(GENERATION_REPO, private=True, exist_ok=True)

# Загрузка адаптеров
print("Uploading extraction adapter...")
api.upload_folder(
    folder_path=extraction_dir,
    repo_id=EXTRACTION_REPO,
    commit_message="Upload extraction LoRA adapter",
)

print("Uploading generation adapter...")
api.upload_folder(
    folder_path=generation_dir,
    repo_id=GENERATION_REPO,
    commit_message="Upload generation LoRA adapter",
)

print("\nAdapters uploaded:")
print(f"  {EXTRACTION_REPO}")
print(f"  {GENERATION_REPO}")
print("\nСервис Sphinx скачает их автоматически при старте.")